# Voyexa AI — Model 1: Hidden Gem Classifier
Predicts `Is_Hidden_Gem` from a place's attributes (cost, terrain, tags, popularity, review volume).
Supervised binary classification — real labels already exist in the dataset, so this is genuinely trainable and evaluable (accuracy, precision/recall, confusion matrix, feature importance).

In [1]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report)

DATA_PATH = os.path.join("..", "..", "data", "processed", "places_processed.csv")
ARTIFACTS_DIR = os.path.join("..", "artifacts")
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

df = pd.read_csv(DATA_PATH)
df.shape

(1163, 202)

## Build feature matrix
Uses the multi-hot interest/suitable_for columns plus cost, difficulty, popularity, zone and terrain. `Hidden_Gem_Score` itself is excluded since it's what defines the label — including it would be leakage.

In [2]:
interest_cols = [c for c in df.columns if c.startswith("interest_")]
suitable_cols = [c for c in df.columns if c.startswith("suitable_")]

feature_cols = interest_cols + suitable_cols + [
    "difficulty_score", "activity_cost_avg", "hotel_cost_avg", "food_cost_avg",
    "total_daily_cost_avg", "popularity_rating_scaled", "number_of_reviewpoints_scaled",
    "Entry_Fee_INR",
]

zone_dummies = pd.get_dummies(df["Zone"], prefix="zone")
terrain_dummies = pd.get_dummies(df["Terrain_Type"], prefix="terrain")

X = pd.concat([df[feature_cols], zone_dummies, terrain_dummies], axis=1)
y = df["Is_Hidden_Gem"].astype(int)

print(f"Features: {X.shape[1]}, Rows: {X.shape[0]}")
y.value_counts()

Features: 187, Rows: 1163


Is_Hidden_Gem
0    924
1    239
Name: count, dtype: int64

## Train / test split + Random Forest classifier

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, class_weight="balanced")
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

## Evaluate

In [4]:
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.3f}")
print(f"Precision: {precision_score(y_test, y_pred):.3f}")
print(f"Recall:    {recall_score(y_test, y_pred):.3f}")
print(f"F1:        {f1_score(y_test, y_pred):.3f}")
print("\nConfusion matrix:\n", confusion_matrix(y_test, y_pred))
print("\n", classification_report(y_test, y_pred))

Accuracy:  0.888
Precision: 0.690
Recall:    0.833
F1:        0.755

Confusion matrix:
 [[167  18]
 [  8  40]]

               precision    recall  f1-score   support

           0       0.95      0.90      0.93       185
           1       0.69      0.83      0.75        48

    accuracy                           0.89       233
   macro avg       0.82      0.87      0.84       233
weighted avg       0.90      0.89      0.89       233



## Feature importance
What actually drives a place being labeled a hidden gem.

In [5]:
importances = pd.Series(clf.feature_importances_, index=X.columns).sort_values(ascending=False)
importances.head(10)

popularity_rating_scaled         0.300257
number_of_reviewpoints_scaled    0.087690
activity_cost_avg                0.050000
suitable_family                  0.041953
total_daily_cost_avg             0.041612
hotel_cost_avg                   0.035051
food_cost_avg                    0.031542
interest_trekking                0.026744
Entry_Fee_INR                    0.026246
suitable_pilgrims                0.025586
dtype: float64

## Save model artifacts

In [6]:
joblib.dump(clf, os.path.join(ARTIFACTS_DIR, "hidden_gem_classifier.pkl"))
joblib.dump(list(X.columns), os.path.join(ARTIFACTS_DIR, "hidden_gem_features.pkl"))
print("Saved model + feature list to ml/artifacts/")

Saved model + feature list to ml/artifacts/
